In [24]:
!pip -q install pvlib scipy
import numpy as np
import pandas as pd
from PIL import Image
from scipy.ndimage import distance_transform_edt

In [25]:
LAT, LON = 38.50, 27.05          # Çiğli civarı (kendi bölgene göre değiştir)
L_STD = 45.0                     # UTC+3 standart meridyen
UTC_OFFSET = 3                   # saat

PANEL_VERIM = 0.20               # r
PANEL_BOY_M = 1.7                # eğimli yöndeki panel boyu L (sıra aralığı için)
ALBEDO = 0.20                    # 0 yaparsan yerden yansıma terimi kapanır

KENAR_M = {"okul": 1.2, "hastane": 1.2, "universite": 1.2, "üniversite": 1.2}
KENAR_VARSAYILAN_M = 2.0         # sanayi vb.

DUZ_EGIMLER = (10, 15)           # düz çatı senaryoları (°)
EGIMLI_CATI_EGIM = 20            # eğimli çatı (°), çatıya paralel
ENGEL_PAYI = (0.70, 0.80)        # kullanılabilir oran alt/üst (Ni vd. 2024 -> ~0,80 merkez; ekip kararı)
GOLGE_SAATI = 10.0               # 21 Aralık, gölgesiz olması gereken en erken güneş saati

PR = 0.80                        # sadece "gerçekçi" sütun için (inverter, kablo, kirlenme)
SICAKLIK_KATSAYI = -0.004        # 1/°C
HUCRE_ISINMA = 0.03              # T_hücre = T_hava + 0.03*G  (basit yaklaşım, varsayım)

In [26]:
# ----------------------------------------------------------------- GÜNEŞ GEOMETRİSİ
def eot_dk(n):
    """Zaman denklemi (dk), Spencer/Duffie-Beckman yaklaşımı."""
    B = np.radians(360 / 365 * (n - 81))
    return 9.87 * np.sin(2 * B) - 7.53 * np.cos(B) - 1.5 * np.sin(B)


def gunes_konumu(index_utc):
    """Her saat için δ, ω ve cosθz. Saat ortası (t+0,5) kullanılır, çünkü TMY değeri o saatin ortalamasıdır."""
    n = index_utc.dayofyear.values
    t_saat = (index_utc.hour.values + 0.5 + UTC_OFFSET) % 24            # duvar saati
    t_gunes = t_saat + (4 * (LON - L_STD) + eot_dk(n)) / 60             # güneş saati
    delta = np.radians(23.45 * np.sin(np.radians(360 / 365 * (284 + n))))
    omega = np.radians(15 * (t_gunes - 12))
    phi = np.radians(LAT)
    cos_zen = np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.cos(omega)
    return delta, omega, cos_zen


def cos_teta(delta, omega, beta_deg, gamma_deg):
    """PDF'teki geliş açısı formülü. β eğim, γ yön (güney 0, doğu -90, batı +90)."""
    phi = np.radians(LAT)
    b, g = np.radians(beta_deg), np.radians(gamma_deg)
    return (np.sin(delta) * np.sin(phi) * np.cos(b)
            - np.sin(delta) * np.cos(phi) * np.sin(b) * np.cos(g)
            + np.cos(delta) * np.cos(phi) * np.cos(b) * np.cos(omega)
            + np.cos(delta) * np.sin(phi) * np.sin(b) * np.cos(g) * np.cos(omega)
            + np.cos(delta) * np.sin(b) * np.sin(g) * np.sin(omega))


def poa(isin, geo, beta, gamma):
    """Eğimli yüzeye düşen ışınım serisi (W/m²)."""
    delta, omega, cos_zen = geo
    ct = np.maximum(0, cos_teta(delta, omega, beta, gamma))
    b = np.radians(beta)
    g = isin.dni.values * ct + isin.dhi.values * (1 + np.cos(b)) / 2
    g = g + ALBEDO * isin.ghi.values * (1 - np.cos(b)) / 2
    return np.where(cos_zen > 0, g, 0.0)



In [27]:
# ----------------------------------------------------------------- IŞINIM VERİSİ
def isinim_getir(offline=False):
    """PVGIS TMY (DNI, DHI, GHI, T_hava). offline=True ise sadece TEST için açık gök modeli."""
    try:
        import pvlib
        if not offline:
            veri = pvlib.iotools.get_pvgis_tmy(LAT, LON, map_variables=True)[0]
            veri.index = pd.to_datetime(veri.index)
            veri.index = veri.index.tz_convert("UTC") if veri.index.tz else veri.index.tz_localize("UTC")
            return veri[["ghi", "dni", "dhi", "temp_air"]]

        idx = pd.date_range("2023-01-01", "2023-12-31 23:00", freq="h", tz="UTC")
        loc = pvlib.location.Location(LAT, LON)
        cs = loc.get_clearsky(idx + pd.Timedelta(minutes=30))
        cs.index = idx
        cs["temp_air"] = 18.0
        return cs[["ghi", "dni", "dhi", "temp_air"]]
    except ModuleNotFoundError:
        # pvlib yoksa yedek çevrimdışı sentetik veri üreteci
        print("Bilgi: 'pvlib' kurulu olmadığı için yedek matematiksel açık gök modeli kullanılıyor.")
        idx = pd.date_range("2023-01-01", "2023-12-31 23:00", freq="h", tz="UTC")
        n = idx.dayofyear.values

        # Güneş saati ortasını hesaplama
        t_saat = (idx.hour.values + 0.5 + UTC_OFFSET) % 24
        t_gunes = t_saat + (4 * (LON - L_STD) + (9.87 * np.sin(2 * np.radians(360 / 365 * (n - 81))) - 7.53 * np.cos(np.radians(360 / 365 * (n - 81))))) / 60
        delta = np.radians(23.45 * np.sin(np.radians(360 / 365 * (284 + n))))
        omega = np.radians(15 * (t_gunes - 12))
        phi = np.radians(LAT)
        cos_zen = np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.cos(omega)

        # Basit ışınım modeli üretimi
        ghi = np.maximum(0, 1000 * cos_zen)
        dni = np.where(cos_zen > 0, 700 * (cos_zen**0.5), 0.0)
        dhi = np.maximum(0, ghi - dni * cos_zen)

        cs = pd.DataFrame({
            "ghi": ghi,
            "dni": dni,
            "dhi": dhi,
            "temp_air": 18.0
        }, index=idx)
        return cs

In [28]:
# ----------------------------------------------------------------- ALAN HESABI
def net_alan_m2(maske_yolu, yer_m_px, kenar_m):
    """Maskeyi kenar boşluğu kadar içeri aşındırıp net alanı verir."""
    m = np.array(Image.open(maske_yolu)) > 0
    d = distance_transform_edt(m) * yer_m_px
    return float((d >= kenar_m).sum() * yer_m_px ** 2)


def zemin_kaplama_orani(beta_deg, gamma_deg=0.0):
    """21 Aralık GOLGE_SAATI'nde sıra gölgesi düşmeyen aralıkla GCR = L cosβ / D."""
    delta = np.radians(-23.45)
    phi = np.radians(LAT)
    omega = np.radians(15 * (GOLGE_SAATI - 12))
    sin_alt = np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.cos(omega)
    alt = np.arcsin(sin_alt)
    cos_az = (sin_alt * np.sin(phi) - np.sin(delta)) / (np.cos(alt) * np.cos(phi))
    az = np.arccos(np.clip(cos_az, -1, 1))                              # güneyden sapma
    b = np.radians(beta_deg)
    golge_uzun = PANEL_BOY_M * np.sin(b) / np.tan(alt) * np.cos(az)     # sıralar yönündeki gölge
    D = PANEL_BOY_M * np.cos(b) + max(golge_uzun, 0)
    return float(PANEL_BOY_M * np.cos(b) / D)


def enerji_kwh(poa_seri, panel_alan_m2, dt_saat=1.0):
    """E = (A*r/1000) * Σ G_POA * Δt"""
    return panel_alan_m2 * PANEL_VERIM / 1000 * float(poa_seri.sum()) * dt_saat


def enerji_gercek_kwh(poa_seri, isin, panel_alan_m2):
    """Sıcaklık kaybı + PR eklenmiş hali (PDF'in 'ileride eklenebilir' kısmı)."""
    t_hucre = isin.temp_air.values + HUCRE_ISINMA * poa_seri
    sicaklik = 1 + SICAKLIK_KATSAYI * (t_hucre - 25)
    return panel_alan_m2 * PANEL_VERIM / 1000 * float((poa_seri * sicaklik).sum()) * PR



In [29]:
# ----------------------------------------------------------------- ANA HESAP
def hesapla(sonuc_csv="sam_sonuc_kamu.csv", kamu_csv="kamu.csv", ozellik_csv=None,
            cikti_csv="cati_enerji_sonuc.csv", offline=False):
    s = pd.read_csv(sonuc_csv)
    k = pd.read_csv(kamu_csv)[["cati_id", "yer_m_px"]]
    d = s.merge(k, on="cati_id", how="left")
    if ozellik_csv:
        d = d.merge(pd.read_csv(ozellik_csv)[["cati_id", "cati_tipi", "azimut"]], on="cati_id", how="left")
    if "cati_tipi" not in d: d["cati_tipi"] = "duz"
    if "azimut" not in d: d["azimut"] = 0.0
    d["cati_tipi"] = d.cati_tipi.fillna("duz")
    d["azimut"] = d.azimut.fillna(0.0)

    isin = isinim_getir(offline)
    geo = gunes_konumu(isin.index)
    onbellek = {}

    def seri(beta, gamma):
        key = (beta, round(gamma / 5) * 5)
        if key not in onbellek:
            onbellek[key] = poa(isin, geo, beta, key[1])
        return onbellek[key]

    satirlar = []
    for r in d.itertuples():
        kenar = KENAR_M.get(str(r.tur).lower(), KENAR_VARSAYILAN_M)
        net = net_alan_m2(r.maske_dosyasi, r.yer_m_px, kenar)
        sonuclar = []   # (kwp, kwh_teorik, kwh_gercek) senaryo başına
        if r.cati_tipi == "duz":
            for beta in DUZ_EGIMLER:
                gcr = zemin_kaplama_orani(beta)
                g = seri(beta, 0.0)                        # düz çatıda paneller güneye
                for eng in ENGEL_PAYI:
                    # GCR = L*cosβ/D yatay izdüşüm oranı; panel alanı = yatay alan * GCR / cosβ
                    A = net * eng * gcr / np.cos(np.radians(beta))
                    sonuclar.append((A * PANEL_VERIM, enerji_kwh(g, A), enerji_gercek_kwh(g, isin, A)))
        else:
            g = seri(EGIMLI_CATI_EGIM, r.azimut)
            for eng in ENGEL_PAYI:
                A = net / np.cos(np.radians(EGIMLI_CATI_EGIM)) * eng
                sonuclar.append((A * PANEL_VERIM, enerji_kwh(g, A), enerji_gercek_kwh(g, isin, A)))
        a = np.array(sonuclar)
        satirlar.append(dict(
            cati_id=r.cati_id, tur=r.tur, cati_tipi=r.cati_tipi, azimut=r.azimut,
            ayak_izi_m2=getattr(r, "ayak_izi_m2", np.nan), sam_alan_m2=r.sam_alan_m2, net_alan_m2=round(net, 1),
            kwp_alt=round(a[:, 0].min(), 1), kwp_ust=round(a[:, 0].max(), 1),
            kwh_teorik_alt=round(a[:, 1].min()), kwh_teorik_ust=round(a[:, 1].max()),
            kwh_gercek_alt=round(a[:, 2].min()), kwh_gercek_ust=round(a[:, 2].max()),
            supheli=getattr(r, "supheli", False)))
    out = pd.DataFrame(satirlar)
    out.to_csv(cikti_csv, index=False)
    return out


def dogrulama_pvlib(offline=False):
    """Formülümüzü pvlib'in izotropik modeliyle karşılaştırır (güney, 15° için)."""
    try:
        import pvlib
    except ModuleNotFoundError:
        print("Bilgi: 'pvlib' kütüphanesi kurulu olmadığı için pvlib doğrulama adımı atlanıyor.")
        return None, None

    isin = isinim_getir(offline)
    geo = gunes_konumu(isin.index)
    bizim = poa(isin, geo, 15, 0.0)
    global ALBEDO
    eski, ALBEDO = ALBEDO, 0.0
    bizim0 = poa(isin, geo, 15, 0.0); ALBEDO = eski
    t = isin.index + pd.Timedelta(minutes=30)
    sp = pvlib.solarposition.get_solarposition(t, LAT, LON)
    sp.index = isin.index
    ref = pvlib.irradiance.get_total_irradiance(15, 180, sp.apparent_zenith, sp.azimuth,
                                                isin.dni, isin.ghi, isin.dhi, model="isotropic")
    print("Yıllık POA (kWh/m²)  bizim: %.0f | pvlib izotropik (albedo'suz): %.0f"
          % (bizim0.sum() / 1000, ref.poa_global.fillna(0).sum() / 1000))
    return bizim0, ref


if __name__ == "__main__":
    import os
    import zipfile

    hedef_dizin = "/content/veri"
    if not os.path.exists(hedef_dizin):
        os.makedirs(hedef_dizin)

    # Mevcut zip arşivlerini hedef /content/veri dizinine çıkartıyoruz
    zipler = [
        "/content/CatiGunes_SAM_Kamu.zip",
        "/content/harita_ek_kamu.zip",
        "/content/sam_sonuc_kamu.zip"
    ]
    for zip_yolu in zipler:
        if os.path.exists(zip_yolu):
            print(f"{zip_yolu} çıkartılıyor...")
            with zipfile.ZipFile(zip_yolu, "r") as z:
                z.extractall(hedef_dizin)

    os.chdir(hedef_dizin)
    dogrulama_pvlib(offline=False) # pvlib kurulumu yoksa çevrimdışı çalıştırılabili
    sonuc = hesapla(sonuc_csv="sam_sonuc_kamu.csv", ozellik_csv="cati_ozellik.csv" if os.path.exists("cati_ozellik.csv") else None, offline=False)
    print(sonuc.groupby("tur")[["net_alan_m2", "kwp_alt", "kwp_ust", "kwh_teorik_alt", "kwh_teorik_ust",
                                "kwh_gercek_alt", "kwh_gercek_ust"]].sum().round())
    print("Toplam gerçekçi üretim bandı (MWh/yıl): %.0f - %.0f"
          % (sonuc.kwh_gercek_alt.sum() / 1000, sonuc.kwh_gercek_ust.sum() / 1000))

/content/CatiGunes_SAM_Kamu.zip çıkartılıyor...
/content/harita_ek_kamu.zip çıkartılıyor...
/content/sam_sonuc_kamu.zip çıkartılıyor...
Yıllık POA (kWh/m²)  bizim: 1985 | pvlib izotropik (albedo'suz): 2001
            net_alan_m2  kwp_alt  kwp_ust  kwh_teorik_alt  kwh_teorik_ust  \
tur                                                                         
hastane         19019.0   1748.0   2239.0         3480539         4362495   
okul            24661.0   2266.0   2903.0         4513202         5656827   
universite      87501.0   8040.0  10302.0        16013186        20070863   

            kwh_gercek_alt  kwh_gercek_ust  
tur                                         
hastane            2585915         3243975  
okul               3353148         4206451  
universite        11897225        14924803  
Toplam gerçekçi üretim bandı (MWh/yıl): 17836 - 22375


In [30]:
import pandas as pd
import os

sonuc_yolu = "/content/veri/cati_enerji_sonuc.csv"
if os.path.exists(sonuc_yolu):
    df_sonuc = pd.read_csv(sonuc_yolu)
    display(df_sonuc.head(20))
else:
    print("Hata: 'cati_enerji_sonuc.csv' dosyası bulunamadı. Lütfen önce hesaplama hücresini çalıştırın.")

,cati_id,tur,cati_tipi,azimut,ayak_izi_m2,sam_alan_m2,net_alan_m2,kwp_alt,kwp_ust,kwh_teorik_alt,kwh_teorik_ust,kwh_gercek_alt,kwh_gercek_ust,supheli
0,1397,okul,duz,0.0,1021.3,893.1,737.2,67.7,86.8,134913,169100,100236,125743,False
1,1398,okul,duz,0.0,1231.5,794.7,631.9,58.1,74.4,115643,144946,85918,107783,True
2,1399,okul,duz,0.0,682.3,538.0,422.6,38.8,49.7,77332,96927,57455,72076,False
3,1400,okul,duz,0.0,325.6,321.1,241.9,22.2,28.5,44278,55498,32897,41269,False
4,1401,okul,duz,0.0,598.2,566.9,450.1,41.4,53.0,82369,103240,61197,76770,False
5,1402,hastane,duz,0.0,5959.1,5353.6,4596.0,422.3,541.1,841092,1054222,624901,783925,True
6,1403,hastane,duz,0.0,553.9,524.9,402.7,37.0,47.4,73690,92363,54749,68682,False
7,1404,okul,duz,0.0,1808.0,1675.8,1460.8,134.2,172.0,267328,335068,198615,249159,False
8,1406,okul,duz,0.0,452.2,409.5,310.7,28.6,36.6,56863,71272,42247,52998,True
9,1407,okul,duz,0.0,817.9,753.3,583.1,53.6,68.7,106717,133759,79287,99464,False
